In [ ]:
!pip install ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive
import gc

# ================================
# 1. Setup
# ================================
drive.mount('/content/drive')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load models
bike_model = YOLO("/content/drive/MyDrive/MP/trained_models/twowheeler.pt").to(device)
rider_model = YOLO("/content/drive/MyDrive/MP/trained_models/rider.pt").to(device)

input_folder = "/content/drive/MyDrive/MP/Test_Images"
output_folder = "/content/drive/MyDrive/MP/output/STAGE-1(FINAL)"
os.makedirs(output_folder, exist_ok=True)

# ✅ DRAWING CONSTANTS (SAME AS REFERENCE)
BOX_THICKNESS = 2
FONT_SCALE = 0.5
FONT_THICKNESS = 2

# ✅ DETECTION CONFIDENCE
INITIAL_BIKE_CONF = 0.15
INITIAL_RIDER_CONF = 0.10

# ================================
# 2. Image Pre-processing Function
# ================================
def enhance_night_image(image):
    """Convert to LAB color space to sharpen only the lightness channel"""
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

# ================================
# 3. Helper Functions
# ================================
def calculate_iou(box1, box2):
    """Calculate Intersection over Union between two boxes"""
    x1, y1, x2, y2 = box1
    x1p, y1p, x2p, y2p = box2
    xi1, yi1 = max(x1, x1p), max(y1, y1p)
    xi2, yi2 = min(x2, x2p), min(y2, y2p)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area1 = (x2 - x1) * (y2 - y1)
    area2 = (x2p - x1p) * (y2p - y1p)
    return inter / (area1 + area2 - inter + 1e-6)

def aggressively_filter_duplicates(detections, iou_threshold=0.40):
    """Remove duplicate detections based on IoU threshold"""
    if not detections:
        return []

    detections = sorted(detections, key=lambda x: x.get('conf', 1.0), reverse=True)

    keep = []
    for det in detections:
        is_duplicate = False
        for kept_det in keep:
            iou = calculate_iou(det['box'], kept_det['box'])
            if iou > iou_threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            keep.append(det)

    return keep

def calculate_distance_center_to_box(point_center, box):
    """Calculate distance from point center to box center"""
    box_cx = (box[0] + box[2]) / 2
    box_cy = (box[1] + box[3]) / 2
    dist = np.sqrt((point_center[0] - box_cx)**2 + (point_center[1] - box_cy)**2)
    return dist

def is_rider_close_to_bike(rider_box, bike_box, img_shape):
    """Check if rider is close to bike - RELAXED constraints from Stage 2"""
    rider_cx = (rider_box[0] + rider_box[2]) / 2
    rider_cy = (rider_box[1] + rider_box[3]) / 2

    bike_cx = (bike_box[0] + bike_box[2]) / 2
    bike_cy = (bike_box[1] + bike_box[3]) / 2

    bike_width = bike_box[2] - bike_box[0]
    bike_height = bike_box[3] - bike_box[1]

    # RELAXED: Rider can be reasonably far from bike
    horizontal_dist = abs(rider_cx - bike_cx)
    if horizontal_dist > bike_width * 5.0:
        return False

    vertical_dist = abs(rider_cy - bike_cy)
    if vertical_dist > bike_height * 4.0:
        return False

    return True

def associate_riders_with_bikes(riders, bikes, img_shape):
    """Associate each rider with its closest bike"""
    associations = {}

    for rider_idx, rider_box in enumerate(riders):
        rider_cx = (rider_box[0] + rider_box[2]) / 2
        rider_cy = (rider_box[1] + rider_box[3]) / 2
        rider_center = np.array([rider_cx, rider_cy])

        min_distance = float('inf')
        closest_bike_idx = -1

        for bike_idx, bike_box in enumerate(bikes):
            if is_rider_close_to_bike(rider_box, bike_box, img_shape):
                distance = calculate_distance_center_to_box(rider_center, bike_box)
                if distance < min_distance:
                    min_distance = distance
                    closest_bike_idx = bike_idx

        if closest_bike_idx != -1:
            associations[rider_idx] = closest_bike_idx

    return associations

# ================================
# 4. Processing Loop
# ================================
image_paths = [p for p in Path(input_folder).glob("*.*") if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]

print(f"\n🚀 Starting vehicle and rider detection...\n")

for img_path in tqdm(image_paths, desc="Processing Images"):
    original_img = cv2.imread(str(img_path))
    if original_img is None:
        continue

    # Enhance the image for better detection in low light
    processed_img = enhance_night_image(original_img)

    # ================================
    # BIKE DETECTION
    # ================================
    bike_results = bike_model(processed_img, conf=INITIAL_BIKE_CONF, iou=0.45, verbose=False)[0]

    # Extract bikes with confidence
    bikes_raw = [
        {'box': b.xyxy[0].cpu().numpy(), 'conf': float(b.conf[0]) if b.conf is not None else 0.0}
        for b in bike_results.boxes
    ]
    # Filter duplicates
    bikes_filtered = aggressively_filter_duplicates(bikes_raw, iou_threshold=0.45)
    bikes = [b['box'] for b in bikes_filtered]
    bikes_conf = [b['conf'] for b in bikes_filtered]

    # ================================
    # RIDER DETECTION
    # ================================
    rider_results = rider_model(processed_img, conf=INITIAL_RIDER_CONF, iou=0.45, verbose=False)[0]

    # Extract riders with confidence
    riders_raw = [
        {'box': r.xyxy[0].cpu().numpy(), 'conf': float(r.conf[0]) if r.conf is not None else 0.0}
        for r in rider_results.boxes
    ]
    # Filter duplicates
    riders_filtered = aggressively_filter_duplicates(riders_raw, iou_threshold=0.50)

    # Filter out oversized boxes (>70% image height)
    img_h, img_w = original_img.shape[:2]
    riders = []
    riders_conf_list = []

    for r in riders_filtered:
        box = r['box']
        rider_height = box[3] - box[1]
        if rider_height <= (img_h * 0.7):
            riders.append(box)
            riders_conf_list.append(r['conf'])

    # ================================
    # ASSOCIATION LOGIC
    # ================================
    rider_to_bike = associate_riders_with_bikes(riders, bikes, original_img.shape)

    # Create reverse mapping (bikes to riders)
    bike_to_riders = {}
    for bike_idx in range(len(bikes)):
        bike_to_riders[bike_idx] = [r_idx for r_idx, b_idx in rider_to_bike.items() if b_idx == bike_idx]

    # ================================
    # DRAWING - SIMPLE APPROACH (LIKE REFERENCE CODE)
    # ================================

    # Draw Bikes (Blue) with confidence
    for bike_idx, bike_box in enumerate(bikes):
        x1, y1, x2, y2 = map(int, bike_box)
        conf = bikes_conf[bike_idx]

        cv2.rectangle(original_img, (x1, y1), (x2, y2), (255, 0, 0), BOX_THICKNESS)
        cv2.putText(original_img, f"Bike {conf:.2f}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, (255, 0, 0), FONT_THICKNESS)

    # Draw Riders (Green) with confidence - only if associated with bike
    for rider_idx, rider_box in enumerate(riders):
        if rider_idx in rider_to_bike:
            x1, y1, x2, y2 = map(int, rider_box)
            conf = riders_conf_list[rider_idx]

            cv2.rectangle(original_img, (x1, y1), (x2, y2), (0, 255, 0), BOX_THICKNESS)
            cv2.putText(original_img, f"Rider {conf:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, (0, 255, 0), FONT_THICKNESS)

    # ================================
    # COUNT STATISTICS
    # ================================
    bikes_count = len([r for r in bike_to_riders.values() if len(r) > 0])
    riders_count = len(rider_to_bike)

    # ================================
    # DISPLAY COUNTS (MINIMALISTIC)
    # ================================
    count_text = f"Bikes: {bikes_count} | Riders: {riders_count}"

    # Get text size for positioning
    text_size = cv2.getTextSize(count_text, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, FONT_THICKNESS)[0]

    # Position at top-left corner with small margin
    x = 10
    y = 20

    # Draw white text
    cv2.putText(original_img, count_text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                FONT_SCALE, (255, 255, 255), FONT_THICKNESS)

    # ================================
    # SAVE OUTPUT
    # ================================
    cv2.imwrite(os.path.join(output_folder, img_path.name), original_img)
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Processing complete!")
print(f"✅ Results saved to: {output_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 Starting vehicle and rider detection...



Processing Images: 100%|██████████| 15/15 [00:06<00:00,  2.44it/s]


✅ Processing complete!
✅ Results saved to: /content/drive/MyDrive/MP/output/STAGE-1(FINAL)


In [ ]:
# ================================
# *******SECOND STAGE********

# IMPORTS & MOUNT
# ================================
import os, cv2, gc, torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive

drive.mount('/content/drive')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ================================
# 1. Setup & Model Loading
# ================================
BIKE_PATH   = "/content/drive/MyDrive/MP/trained_models/twowheeler.pt"
RIDER_PATH  = "/content/drive/MyDrive/MP/trained_models/rider.pt"
HELMET_PATH = "/content/drive/MyDrive/MP/trained_models/helmet.pt"

input_folder  = "/content/drive/MyDrive/MP/Test_Images"
output_folder = "/content/drive/MyDrive/MP/output/STAGE-2(FINAL)"
os.makedirs(output_folder, exist_ok=True)

# ✅ FIXED CONSTANTS - CONSISTENT ACROSS ALL IMAGES
BOX_THICKNESS = 5  # INCREASED from 3 to 5 for better visibility
FONT_SCALE = 0.7  # Consistent font scale
FONT_THICKNESS = 2

# ✅ CORNER TEXT CONSTANTS - RESPONSIVE TO IMAGE SIZE FOR CONSISTENCY
# These are BASE values that scale with image height
BASE_CORNER_FONT_SCALE = 1.0  # Base font scale (scales with image)
BASE_CORNER_FONT_THICKNESS = 3  # Base thickness (scales with image)
BASE_CORNER_BOX_PADDING = 20  # Base padding (scales with image)
BASE_CORNER_MARGIN = 20  # Base margin from corner (scales with image)

def load_all():
    bike_m   = YOLO(BIKE_PATH).to(device)
    rider_m  = YOLO(RIDER_PATH).to(device)
    helmet_m = YOLO(HELMET_PATH).to(device)
    print(f"✅ All YOLO models loaded on {device}")
    return bike_m, rider_m, helmet_m

bike_model, rider_model, helmet_model = load_all()

# ================================
# 2. Helpers
# ================================
def calculate_iou(box1, box2):
    x1, y1, x2, y2 = box1
    x1p, y1p, x2p, y2p = box2
    xi1, yi1 = max(x1, x1p), max(y1, y1p)
    xi2, yi2 = min(x2, x2p), min(y2, y2p)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    area1 = (x2 - x1) * (y2 - y1)
    area2 = (x2p - x1p) * (y2p - y1p)
    return inter / (area1 + area2 - inter + 1e-6)

def aggressively_filter_duplicates(detections, iou_threshold=0.40):
    """
    Remove duplicate detections more aggressively to reduce overfitting
    """
    if not detections:
        return []

    # Sort by confidence (descending)
    detections = sorted(detections, key=lambda x: x.get('conf', 1.0), reverse=True)

    keep = []
    for det in detections:
        is_duplicate = False
        for kept_det in keep:
            iou = calculate_iou(det['box'], kept_det['box'])
            if iou > iou_threshold:
                is_duplicate = True
                break

        if not is_duplicate:
            keep.append(det)

    return keep

def is_rider_close_to_bike(rider_box, bike_box, img_shape):
    """Check if rider is close to bike - VERY RELAXED constraints"""
    rider_cx = (rider_box[0] + rider_box[2]) / 2
    rider_cy = (rider_box[1] + rider_box[3]) / 2

    bike_cx = (bike_box[0] + bike_box[2]) / 2
    bike_cy = (bike_box[1] + bike_box[3]) / 2

    bike_width = bike_box[2] - bike_box[0]
    bike_height = bike_box[3] - bike_box[1]

    # VERY RELAXED: Rider can be very far from bike
    horizontal_dist = abs(rider_cx - bike_cx)
    if horizontal_dist > bike_width * 5.0:  # Much more relaxed (was 2.5)
        return False

    vertical_dist = abs(rider_cy - bike_cy)
    if vertical_dist > bike_height * 4.0:  # Much more relaxed (was 2.0)
        return False

    return True

def calculate_distance_center_to_box(point_center, box):
    """Calculate distance from point center to box center"""
    box_cx = (box[0] + box[2]) / 2
    box_cy = (box[1] + box[3]) / 2

    dist = np.sqrt((point_center[0] - box_cx)**2 + (point_center[1] - box_cy)**2)
    return dist

def associate_riders_with_bikes(riders, bikes, img_shape):
    """Associate each rider with its closest bike"""
    associations = {}

    for rider_idx, rider_box in enumerate(riders):
        rider_cx = (rider_box[0] + rider_box[2]) / 2
        rider_cy = (rider_box[1] + rider_box[3]) / 2
        rider_center = np.array([rider_cx, rider_cy])

        min_distance = float('inf')
        closest_bike_idx = -1

        for bike_idx, bike_box in enumerate(bikes):
            if is_rider_close_to_bike(rider_box, bike_box, img_shape):
                distance = calculate_distance_center_to_box(rider_center, bike_box)
                if distance < min_distance:
                    min_distance = distance
                    closest_bike_idx = bike_idx

        if closest_bike_idx != -1:
            associations[rider_idx] = closest_bike_idx

    return associations

def associate_helmets_with_riders(helmets, riders):
    """Associate each helmet with its closest rider"""
    associations = {}

    for helmet_idx, helmet_box in enumerate(helmets):
        helmet_cx = (helmet_box['box'][0] + helmet_box['box'][2]) / 2
        helmet_cy = (helmet_box['box'][1] + helmet_box['box'][3]) / 2
        helmet_center = np.array([helmet_cx, helmet_cy])

        min_distance = float('inf')
        closest_rider_idx = -1

        for rider_idx, rider_box in enumerate(riders):
            rider_height = rider_box[3] - rider_box[1]
            rider_middle = rider_box[1] + rider_height * 0.5

            if helmet_cy > rider_middle:
                continue

            rider_cx = (rider_box[0] + rider_box[2]) / 2
            rider_width = rider_box[2] - rider_box[0]

            horizontal_dist = abs(helmet_cx - rider_cx)
            if horizontal_dist > rider_width * 1.0:
                continue

            distance = calculate_distance_center_to_box(helmet_center, rider_box)
            if distance < min_distance:
                min_distance = distance
                closest_rider_idx = rider_idx

        if closest_rider_idx != -1:
            associations[helmet_idx] = closest_rider_idx

    return associations

# ✅ NEW HELPER: Draw corner text with RESPONSIVE scaling based on image size
def draw_corner_text(canvas, text, position='top_left', bg_color=(0, 0, 255), border_color=(255, 165, 0)):
    """
    Draw responsive corner text that scales with image size
    position: 'top_left', 'top_right', 'bottom_left', 'bottom_right'

    The text scales proportionally with image height to maintain visual consistency
    across different image resolutions.
    """
    h, w = canvas.shape[:2]

    # ✅ CALCULATE SCALE FACTOR based on image height
    # Reference height is 1080p, scale linearly from there
    reference_height = 1080
    scale_factor = h / reference_height

    # Ensure minimum scale even for small images
    scale_factor = max(scale_factor, 0.6)
    # Cap maximum scale for very large images
    scale_factor = min(scale_factor, 2.0)

    # ✅ APPLY SCALING to all constants
    font_scale = BASE_CORNER_FONT_SCALE * scale_factor
    font_thickness = max(2, int(BASE_CORNER_FONT_THICKNESS * scale_factor))
    padding = int(BASE_CORNER_BOX_PADDING * scale_factor)
    margin = int(BASE_CORNER_MARGIN * scale_factor)

    # Get text size with scaled font
    text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)[0]
    text_width = text_size[0]
    text_height = text_size[1]

    # Calculate box dimensions with scaled padding
    box_width = text_width + (padding * 2)
    box_height = text_height + (padding)

    # Calculate position based on corner and margin
    if position == 'top_left':
        x1, y1 = margin, margin
        x2, y2 = x1 + box_width, y1 + box_height
        text_x = x1 + padding
        text_y = y1 + padding + text_height - 2

    elif position == 'top_right':
        x1 = w - box_width - margin
        y1 = margin
        x2 = x1 + box_width
        y2 = y1 + box_height
        text_x = x1 + padding
        text_y = y1 + padding + text_height - 2

    elif position == 'bottom_left':
        x1 = margin
        y1 = h - box_height - margin
        x2 = x1 + box_width
        y2 = h - margin
        text_x = x1 + padding
        text_y = y2 - padding + 2

    elif position == 'bottom_right':
        x1 = w - box_width - margin
        y1 = h - box_height - margin
        x2 = w - margin
        y2 = h - margin
        text_x = x1 + padding
        text_y = y2 - padding + 2

    # Ensure coordinates are within bounds
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)

    # Draw background box
    cv2.rectangle(canvas, (x1, y1), (x2, y2), bg_color, -1)
    # Draw border
    cv2.rectangle(canvas, (x1, y1), (x2, y2), border_color, 3)
    # Draw text
    cv2.putText(canvas, text, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX,
                font_scale, (0, 255, 255), font_thickness)

# ================================
# 3. Pipeline Logic
# ================================
def run_violation_check(img):
    # Initial confidence levels
    initial_bike_conf = 0.35
    initial_rider_conf = 0.30
    initial_helmet_conf = 0.35

    # Run initial detections
    b_res = bike_model(img, conf=initial_bike_conf, agnostic_nms=True, verbose=False)[0]
    r_res = rider_model(img, conf=initial_rider_conf, agnostic_nms=True, verbose=False)[0]
    h_res = helmet_model(img, conf=initial_helmet_conf, agnostic_nms=True, verbose=False)[0]

    canvas = img.copy()

    # Process bikes - VERY STRICT duplicate removal
    bikes_raw = [{'box': b.xyxy[0].cpu().numpy(), 'conf': float(b.conf[0]) if b.conf is not None else 0.0}
                 for b in b_res.boxes]
    bikes_filtered = aggressively_filter_duplicates(bikes_raw, iou_threshold=0.45)
    bikes = [b['box'] for b in bikes_filtered]

    # --- INTELLIGENT RIDER DETECTION ---
    # Start with high confidence, iteratively reduce if needed
    riders = []
    rider_conf = initial_rider_conf
    confidence_steps = [0.30, 0.25, 0.20, 0.15, 0.10, 0.05]  # Confidence levels to try

    for conf_level in confidence_steps:
        r_res = rider_model(img, conf=conf_level, agnostic_nms=True, verbose=False)[0]

        riders_raw = [{'box': r.xyxy[0].cpu().numpy(), 'conf': float(r.conf[0]) if r.conf is not None else 0.0}
                      for r in r_res.boxes]
        riders_filtered = aggressively_filter_duplicates(riders_raw, iou_threshold=0.50)
        riders = [r['box'] for r in riders_filtered]

        # If we have detections on bikes, use this confidence level
        if len(riders) > 0:
            # Check if we have reasonable detections
            riders_on_bikes = sum(1 for rider in riders for bike in bikes if is_rider_close_to_bike(rider, bike, img.shape))
            if riders_on_bikes > 0 or conf_level == confidence_steps[-1]:  # Use this or if it's the last attempt
                break

    # --- INTELLIGENT HELMET DETECTION ---
    # Adapt helmet confidence based on rider count
    helmets = []

    if len(riders) == 0:
        # No riders detected, use high confidence for helmets
        helmet_conf_to_use = initial_helmet_conf
    else:
        # Riders detected, adjust helmet confidence based on mismatch
        helmet_conf_to_use = initial_helmet_conf

    helmet_conf_steps = [0.35, 0.30, 0.25, 0.20, 0.15]

    for conf_level in helmet_conf_steps:
        h_res = helmet_model(img, conf=conf_level, agnostic_nms=True, verbose=False)[0]

        helmets_raw = []
        for h in h_res.boxes:
            helmets_raw.append({
                'box': h.xyxy[0].cpu().numpy(),
                'label': helmet_model.names[int(h.cls[0])].lower(),
                'conf': float(h.conf[0]) if h.conf is not None else 0.0
            })

        helmets_filtered = aggressively_filter_duplicates(helmets_raw, iou_threshold=0.40)
        helmets = [{'box': h['box'], 'is_violation': "no" in h['label'], 'label': h['label']}
                   for h in helmets_filtered]

        # If we have detections matching riders, use this level
        if abs(len(helmets) - len(riders)) <= 1 or conf_level == helmet_conf_steps[-1]:
            break

    # Return None only if NO detections at all
    if len(riders) == 0 and len(helmets) == 0:
        return None

    # --- VIOLATION TRACKING WITH INTELLIGENT ASSOCIATIONS ---

    # Associate riders with bikes
    rider_to_bike = associate_riders_with_bikes(riders, bikes, img.shape)

    # Associate helmets with riders
    helmet_to_rider = associate_helmets_with_riders(helmets, riders)

    # Create reverse mappings
    bike_to_riders = {}
    for bike_idx in range(len(bikes)):
        bike_to_riders[bike_idx] = [r_idx for r_idx, b_idx in rider_to_bike.items() if b_idx == bike_idx]

    rider_to_helmets = {}
    for rider_idx in range(len(riders)):
        rider_to_helmets[rider_idx] = [h_idx for h_idx, r_idx in helmet_to_rider.items() if r_idx == rider_idx]

    # Check helmet violations
    rider_helmet_violations = {}
    for rider_idx in range(len(riders)):
        rider_helmet_violations[rider_idx] = False

        helmets_on_rider = rider_to_helmets.get(rider_idx, [])

        if len(helmets_on_rider) == 0:
            rider_helmet_violations[rider_idx] = True
        else:
            for h_idx in helmets_on_rider:
                if helmets[h_idx]['is_violation']:
                    rider_helmet_violations[rider_idx] = True
                    break

    helmet_violation_count = sum(1 for v in rider_helmet_violations.values() if v)

    # Check triple riding violations
    triple_riding_violations = []
    for bike_idx, bike_box in enumerate(bikes):
        riders_on_bike = bike_to_riders.get(bike_idx, [])
        rider_count = len(riders_on_bike)

        helmet_count = sum(len(rider_to_helmets.get(r_idx, [])) for r_idx in riders_on_bike)

        if rider_count >= 3:
            triple_riding_violations.append({
                'bike_box': bike_box,
                'rider_count': rider_count,
                'helmet_count': helmet_count,
                'occupant_count': rider_count,
                'bike_idx': bike_idx
            })

    # --- FALLBACK: Detect triple riding by spatial clustering ---
    # If no triple riding found via association, check if riders cluster around bikes
    if len(triple_riding_violations) == 0 and len(riders) >= 3 and len(bikes) > 0:
        # Check if 3+ riders are spatially close to each other (likely on same bike)
        from scipy.spatial.distance import cdist

        try:
            # Get rider centers
            rider_centers = np.array([[(r[0] + r[2])/2, (r[1] + r[3])/2] for r in riders])
            bike_centers = np.array([[(b[0] + b[2])/2, (b[1] + b[3])/2] for b in bikes])

            # For each bike, find nearby riders
            for bike_idx, bike_center in enumerate(bike_centers):
                bike_box = bikes[bike_idx]

                # Calculate distances from riders to this bike center
                distances = np.sqrt(np.sum((rider_centers - bike_center) ** 2, axis=1))

                # Get riders within reasonable distance (relaxed threshold)
                bike_height = bike_box[3] - bike_box[1]
                bike_width = bike_box[2] - bike_box[0]
                threshold = np.sqrt(bike_height**2 + bike_width**2) * 2.0  # 2x diagonal

                nearby_riders = np.where(distances < threshold)[0]

                if len(nearby_riders) >= 3:
                    # Triple riding detected!
                    helmet_count_fallback = sum(len(rider_to_helmets.get(int(r_idx), [])) for r_idx in nearby_riders)

                    triple_riding_violations.append({
                        'bike_box': bike_box,
                        'rider_count': len(nearby_riders),
                        'helmet_count': helmet_count_fallback,
                        'occupant_count': len(nearby_riders),
                        'bike_idx': bike_idx
                    })
                    break  # Only flag one bike per image to avoid duplicates
        except Exception as e:
            pass  # If clustering fails, continue without it

    # Check for bikes without riders
    bikes_without_riders = [bike_idx for bike_idx, riders_list in bike_to_riders.items() if len(riders_list) == 0]
    valid_bike_indices = set(bike_to_riders.keys()) - set(bikes_without_riders)

    # --- DRAWING (SHOW RIDERS + HELMETS) ---

    # Draw rider boxes (GREEN for safe, RED for violation) with FIXED thickness
    for rider_idx, rider_box in enumerate(riders):
        x1, y1, x2, y2 = map(int, rider_box)

        # Check helmet violation status
        has_violation = rider_helmet_violations.get(rider_idx, False)
        color = (0, 0, 255) if has_violation else (0, 255, 0)

        # ✅ USE FIXED BOX_THICKNESS
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, BOX_THICKNESS)

        helmets_on_rider = rider_to_helmets.get(rider_idx, [])
        helmet_count = len(helmets_on_rider)
        text = f"Rider ({helmet_count}H)" if helmet_count > 0 else "Rider (No H)"

        text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, FONT_THICKNESS)[0]
        cv2.rectangle(canvas, (x1, y1 - text_size[1] - 6), (x1 + text_size[0] + 4, y1 - 2), color, -1)
        cv2.putText(canvas, text, (x1 + 2, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, (255, 255, 255), FONT_THICKNESS)

    # Draw helmet boxes with FIXED thickness
    for helmet_idx, helmet in enumerate(helmets):
        hx1, hy1, hx2, hy2 = map(int, helmet['box'])
        color = (0, 0, 255) if helmet['is_violation'] else (0, 255, 0)
        text = "No Helmet: Violation" if helmet['is_violation'] else "Helmet"

        # ✅ USE FIXED BOX_THICKNESS
        cv2.rectangle(canvas, (hx1, hy1), (hx2, hy2), color, BOX_THICKNESS)

        text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, FONT_THICKNESS)[0]
        cv2.rectangle(canvas, (hx1, hy1 - text_size[1] - 6), (hx1 + text_size[0] + 4, hy1 - 2), color, -1)
        cv2.putText(canvas, text, (hx1 + 2, hy1 - 5), cv2.FONT_HERSHEY_SIMPLEX, FONT_SCALE, (255, 255, 255), FONT_THICKNESS)

    # --- ✅ HELMET VIOLATIONS COUNT IN TOP LEFT (FIXED SIZE) ---
    if helmet_violation_count > 0:
        label = f"Helmet Violations: {helmet_violation_count}"
        draw_corner_text(canvas, label, position='top_left', bg_color=(0, 0, 255), border_color=(0, 165, 255))

    # --- ✅ TRIPLE RIDING COUNT IN TOP RIGHT (FIXED SIZE) ---
    if triple_riding_violations:
        violation_count = len(triple_riding_violations)
        label = f"TRIPLE RIDING: {violation_count}"
        draw_corner_text(canvas, label, position='top_right', bg_color=(0, 0, 255), border_color=(255, 165, 0))

    return canvas

# ================================
# 4. Execute
# ================================
print("\n🚀 Starting violation detection...\n")

paths = sorted(list(Path(input_folder).glob("*.*")))

for p in tqdm(paths, desc="Processing Images"):
    img = cv2.imread(str(p))
    if img is None:
        continue

    # Run violation check
    result = run_violation_check(img)

    if result is not None:
        pass
    else:
        result = img

    cv2.imwrite(os.path.join(output_folder, p.name), result)
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Processing complete!")
print("\n✅ All violations processed. System Clean.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All YOLO models loaded on cuda

🚀 Starting violation detection...



Processing Images: 100%|██████████| 10/10 [00:23<00:00,  2.35s/it]


✅ Processing complete!

✅ All violations processed. System Clean.


In [ ]:
# ================================
# 1. INSTALL LATEST SDKs
# ================================
!pip install -q -U google-auth==2.47.0 google-genai easyocr ultralytics

import os, cv2, torch, gc, re, time
import numpy as np
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive
import easyocr
from google import genai
from google.genai import types

# ================================
# 2. MOUNT DRIVE & CONFIG
# ================================
drive.mount('/content/drive', timeout_ms=60000)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 🔑 API CONFIG – REPLACE WITH YOUR ACTUAL API KEY
API_KEY = "YOUR-KEY"          # <-- Change this!
client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-3.0-flash"

# ================================
# 3. PATHS (YOUR FOLDER STRUCTURE)
# ================================
PLATE_PATH   = "/content/drive/MyDrive/MP/trained_models/plate.pt"
input_folder = "/content/drive/MyDrive/MP/Test_Images"
output_folder = "/content/drive/MyDrive/MP/output/STAGE-3(FINAL)"
crop_folder   = "/content/drive/MyDrive/MP/output/STAGE-3(CROPPED PLATES)"

os.makedirs(output_folder, exist_ok=True)
os.makedirs(crop_folder, exist_ok=True)

# ================================
# 4. LOAD MODELS
# ================================
plate_model = YOLO(PLATE_PATH).to(device)
ocr_reader = easyocr.Reader(['en'], gpu=(device=='cuda'))

# ================================
# 5. IMAGE ENHANCEMENT
# ================================
def preprocess_for_blur(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    kernel = np.array([[-1,-1,-1], [-1, 9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(gray, -1, kernel)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(sharpened)

def clean_ocr(text):
    return re.sub(r'[^A-Z0-9]', '', text.upper().strip())

def format_indian_plate(text):
    t = clean_ocr(text)
    if len(t) < 6: return t
    return f"{t[:2]} {t[2:4]} {t[4:-4]} {t[-4:]}"

# ================================
# 6. GEMINI OCR (FALLBACK)
# ================================
def gemini_ocr_v2(crop_path):
    max_retries = 3
    for attempt in range(max_retries):
        try:
            with open(crop_path, "rb") as f:
                img_bytes = f.read()
            prompt = "Read the Indian vehicle number plate. Only output the alphanumeric text."
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=[
                    types.Part.from_bytes(data=img_bytes, mime_type="image/jpeg"),
                    types.Part.from_text(text=prompt)
                ]
            )
            return clean_ocr(response.text)
        except Exception as e:
            if "429" in str(e):
                print(f"⚠️ Rate limit hit. Waiting 35s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(35)
                continue
            else:
                print(f"Gemini API Error: {e}")
                return ""
    return ""

# ================================
# 7. HYBRID OCR LOGIC
# ================================
def run_ocr(crop_path):
    img = cv2.imread(crop_path)
    if img is None: return "UNREAD"

    best_text = ""
    best_score = 0
    rotations = {0: None, 90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}

    for angle, code in rotations.items():
        rot = img if code is None else cv2.rotate(img, code)
        proc = preprocess_for_blur(rot)
        results = ocr_reader.readtext(proc, contrast_ths=0.1, adjust_contrast=0.7)
        if results:
            text = "".join([r[1] for r in results])
            score = sum([r[2] for r in results])
            if len(clean_ocr(text)) > len(clean_ocr(best_text)) or score > best_score:
                best_text, best_score = text, score

    final_text = clean_ocr(best_text)
    if len(final_text) < 6:
        gem_text = gemini_ocr_v2(crop_path)
        if len(gem_text) > len(final_text):
            final_text = gem_text

    return format_indian_plate(final_text) if len(final_text) >= 3 else "UNREAD"

# ================================
# 8. MAIN LOOP – NOW SAVES ANNOTATED IMAGES
# ================================
print(f"\n🚀 Running Pipeline (Model: {MODEL_ID})...\n")
paths = list(Path(input_folder).glob("*.*"))

for p in tqdm(paths):
    img = cv2.imread(str(p))
    if img is None:
        continue

    # Make a copy to draw on
    annotated_img = img.copy()
    detections = plate_model(img, conf=0.3)[0]

    for i, box in enumerate(detections.boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        # Save cropped plate (optional)
        temp_crop_path = os.path.join(crop_folder, f"crop_{p.stem}_{i}.jpg")
        cv2.imwrite(temp_crop_path, crop)

        # Get OCR text
        result_text = run_ocr(temp_crop_path)
        print(f"File: {p.name} | Plate {i}: {result_text}")

        # Draw bounding box and text on the annotated image
        cv2.rectangle(annotated_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        # Put text above the box (or inside if preferred)
        label = f"Plate: {result_text}"
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(annotated_img, (x1, y1 - text_h - 5), (x1 + text_w, y1), (0, 255, 0), -1)
        cv2.putText(annotated_img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        # Small delay to respect free-tier rate limits
        time.sleep(2)

    # Save the annotated image to output_folder
    out_path = os.path.join(output_folder, f"{p.stem}_annotated.jpg")
    cv2.imwrite(out_path, annotated_img)

    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ DONE. Annotated images saved to: {output_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 Running Pipeline (Model: gemini-3.0-flash)...



  0%|          | 0/10 [00:00<?, ?it/s]


0: 640x384 2 number_plates, 10.5ms
Speed: 2.5ms preprocess, 10.5ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)
Gemini API Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3.0-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
File: test_image_02.jpg | Plate 0: UNREAD
Gemini API Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3.0-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
File: test_image_02.jpg | Plate 1: UNREAD


 10%|█         | 1/10 [00:05<00:45,  5.01s/it]


0: 640x384 3 number_plates, 9.2ms
Speed: 3.0ms preprocess, 9.2ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)
Gemini API Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3.0-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
File: test_image_03.jpg | Plate 0: UNREAD
Gemini API Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3.0-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
File: test_image_03.jpg | Plate 1: UNREAD
Gemini API Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3.0-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and th

...


0: 640x384 1 number_plate, 9.7ms
Speed: 2.8ms preprocess, 9.7ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 384)
Gemini API Error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3.0-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
File: test_image_10.png | Plate 0: UNREAD


 90%|█████████ | 9/10 [00:37<00:03,  3.50s/it]


0: 480x640 (no detections), 10.0ms
Speed: 2.9ms preprocess, 10.0ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)


100%|██████████| 10/10 [00:38<00:00,  3.80s/it]


✅ DONE. Annotated images saved to: /content/drive/MyDrive/MP/output/STAGE-3(FINAL)


In [ ]:
# ================================
# 1. INSTALL LATEST SDKs
# ================================
!pip install -q -U easyocr ultralytics

import os, cv2, torch, gc, re
import numpy as np
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive
import easyocr

# ================================
# 2. MOUNT DRIVE (FORCE REMOUNT)
# ================================
drive.mount('/content/drive', force_remount=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ================================
# 3. PATHS (YOUR FOLDER STRUCTURE)
# ================================
PLATE_PATH   = "/content/drive/MyDrive/MP/trained_models/plate.pt"
input_folder = "/content/drive/MyDrive/MP/Test_Images"

# Output folders
output_root = "/content/drive/MyDrive/MP/output/STAGE-3(FINAL)"
annotated_folder = os.path.join(output_root, "ocr_image_output")   # all annotated images
crop_folder      = os.path.join(output_root, "crop_plate")         # cropped plates
detected_only_folder = os.path.join(output_root, "detected_plates") # only images with plates

os.makedirs(annotated_folder, exist_ok=True)
os.makedirs(crop_folder, exist_ok=True)
os.makedirs(detected_only_folder, exist_ok=True)

# ================================
# 4. LOAD MODELS
# ================================
plate_model = YOLO(PLATE_PATH).to(device)
ocr_reader = easyocr.Reader(['en'], gpu=(device=='cuda'))

# ================================
# 5. IMAGE ENHANCEMENT & OCR CLEANUP
# ================================
def preprocess_for_blur(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    kernel = np.array([[-1,-1,-1], [-1, 9,-1], [-1,-1,-1]])
    sharpened = cv2.filter2D(gray, -1, kernel)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(sharpened)

def clean_ocr(text):
    return re.sub(r'[^A-Z0-9]', '', text.upper().strip())

def correct_plate_format(raw_text):
    """Indian format: 2 letters + 2 digits + 1-2 letters + 4 digits"""
    raw = clean_ocr(raw_text)
    if len(raw) < 8:
        return raw

    letter_to_digit = {'O':'0', 'I':'1', 'Z':'2', 'S':'5', 'B':'8', 'G':'6'}
    digit_to_letter = {'0':'O', '1':'I', '2':'Z', '5':'S', '8':'B', '6':'G'}

    if len(raw) == 9:  # 2+2+1+4
        corrected = list(raw)
        for i in [0,1]:
            if corrected[i].isdigit():
                corrected[i] = digit_to_letter.get(corrected[i], corrected[i])
        for i in [2,3]:
            if corrected[i].isalpha():
                corrected[i] = letter_to_digit.get(corrected[i], corrected[i])
        if corrected[4].isdigit():
            corrected[4] = digit_to_letter.get(corrected[4], corrected[4])
        for i in range(5, 9):
            if corrected[i].isalpha():
                corrected[i] = letter_to_digit.get(corrected[i], corrected[i])
        corrected_str = ''.join(corrected)
        return f"{corrected_str[:2]} {corrected_str[2:4]} {corrected_str[4]} {corrected_str[5:9]}"

    elif len(raw) == 10:  # 2+2+2+4
        corrected = list(raw)
        for i in [0,1]:
            if corrected[i].isdigit():
                corrected[i] = digit_to_letter.get(corrected[i], corrected[i])
        for i in [2,3]:
            if corrected[i].isalpha():
                corrected[i] = letter_to_digit.get(corrected[i], corrected[i])
        for i in [4,5]:
            if corrected[i].isdigit():
                corrected[i] = digit_to_letter.get(corrected[i], corrected[i])
        for i in range(6, 10):
            if corrected[i].isalpha():
                corrected[i] = letter_to_digit.get(corrected[i], corrected[i])
        corrected_str = ''.join(corrected)
        return f"{corrected_str[:2]} {corrected_str[2:4]} {corrected_str[4:6]} {corrected_str[6:10]}"
    else:
        return raw

def format_indian_plate(text):
    t = clean_ocr(text)
    if len(t) < 6:
        return t
    return correct_plate_format(t)

def run_ocr(crop_path):
    img = cv2.imread(crop_path)
    if img is None:
        return "UNREAD"

    best_text = ""
    best_score = 0
    rotations = {0: None, 90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}

    for angle, code in rotations.items():
        rot = img if code is None else cv2.rotate(img, code)
        proc = preprocess_for_blur(rot)
        results = ocr_reader.readtext(proc, contrast_ths=0.1, adjust_contrast=0.7)
        if results:
            text = "".join([r[1] for r in results])
            score = sum([r[2] for r in results])
            if len(clean_ocr(text)) > len(clean_ocr(best_text)) or score > best_score:
                best_text, best_score = text, score

    final_text = clean_ocr(best_text)
    return format_indian_plate(final_text) if len(final_text) >= 3 else "UNREAD"

# ================================
# 6. MAIN LOOP – SAVE ONLY DETECTED IMAGES TO SEPARATE FOLDER
# ================================
print(f"\n🚀 Running pipeline (no Gemini, only EasyOCR)...\n")
paths = list(Path(input_folder).glob("*.*"))

for p in tqdm(paths):
    img = cv2.imread(str(p))
    if img is None:
        continue

    # Detect plates
    detections = plate_model(img, conf=0.3)[0]
    if len(detections.boxes) == 0:
        print(f"❌ No plate detected in {p.name} – skipping")
        continue   # skip saving this image to detected_plates folder

    # Make a copy to draw on
    annotated_img = img.copy()
    plate_detected = False

    for i, box in enumerate(detections.boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        # Save cropped plate
        temp_crop_path = os.path.join(crop_folder, f"crop_{p.stem}_{i}.jpg")
        cv2.imwrite(temp_crop_path, crop)

        # Get OCR text
        result_text = run_ocr(temp_crop_path)
        print(f"✅ {p.name} | Plate {i}: {result_text}")

        # Draw bounding box and text
        cv2.rectangle(annotated_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"Plate: {result_text}"
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(annotated_img, (x1, y1 - text_h - 5), (x1 + text_w, y1), (0, 255, 0), -1)
        cv2.putText(annotated_img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

        plate_detected = True

    # Save annotated image to both folders
    if plate_detected:
        # 1) Save to the main annotated folder (all detected images)
        out_path_main = os.path.join(annotated_folder, f"{p.stem}_annotated.jpg")
        cv2.imwrite(out_path_main, annotated_img)

        # 2) Save to the "detected_plates" folder (only images with plates)
        out_path_detected = os.path.join(detected_only_folder, f"{p.stem}_annotated.jpg")
        cv2.imwrite(out_path_detected, annotated_img)

    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ DONE.")
print(f"📁 All annotated images (with plates) saved to: {annotated_folder}")
print(f"📁 Only images with plates also saved to: {detected_only_folder}")
print(f"📁 Cropped plates saved to: {crop_folder}")

Mounted at /content/drive

🚀 Running pipeline (no Gemini, only EasyOCR)...



  0%|          | 0/10 [00:00<?, ?it/s]


0: 640x384 2 number_plates, 11.5ms
Speed: 2.6ms preprocess, 11.5ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 384)
✅ test_image_02.jpg | Plate 0: UNREAD
✅ test_image_02.jpg | Plate 1: UNREAD


 10%|█         | 1/10 [00:00<00:08,  1.06it/s]


0: 640x384 3 number_plates, 8.8ms
Speed: 3.2ms preprocess, 8.8ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)
✅ test_image_03.jpg | Plate 0: UNREAD
✅ test_image_03.jpg | Plate 1: UNREAD
✅ test_image_03.jpg | Plate 2: UNREAD


...


0: 640x384 1 number_plate, 10.3ms
Speed: 2.3ms preprocess, 10.3ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 384)
✅ test_image_10.png | Plate 0: UNREAD


 90%|█████████ | 9/10 [00:07<00:00,  1.51it/s]


0: 480x640 (no detections), 9.8ms
Speed: 2.6ms preprocess, 9.8ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)


100%|██████████| 10/10 [00:07<00:00,  1.26it/s]

❌ No plate detected in images_013.jpg – skipping

✅ DONE.
📁 All annotated images (with plates) saved to: /content/drive/MyDrive/MP/output/STAGE-3(FINAL)/ocr_image_output
📁 Only images with plates also saved to: /content/drive/MyDrive/MP/output/STAGE-3(FINAL)/detected_plates
📁 Cropped plates saved to: /content/drive/MyDrive/MP/output/STAGE-3(FINAL)/crop_plate
